# M01. 파일 읽기 + 인코딩

> 📌 **언제 필요한가**  
> CSV/Excel 파일을 받았는데 한글이 깨질 때, `UnicodeDecodeError`가 날 때.

## 이 모듈에서 배울 것

- `pd.read_csv` 기본 사용법
- 한국 공공데이터의 두 가지 표준 인코딩: **cp949**와 **utf-8**
- 인코딩 잘못 잡았을 때의 증상과 해결법
- 인코딩 자동 탐지

---


📥 데이터 준비

이 모듈은 아래 파일이 필요합니다. (받는 곳 링크 → 바로 다운로드)

- `경찰청_범죄 발생 지역별 통계.csv` — 인코딩 `cp949` — [공공데이터포털에서 받기](https://www.data.go.kr/data/3074462/fileData.do)
- `스트레스_인지율.csv` — 인코딩 `utf-8` — [KOSIS에서 받기](https://kosis.kr/statisticsList/statisticsListIndex.do?menuId=M_01_01&vwcd=MT_ZTITLE&parmTabId=M_01_01&parentId=F.1;F_55.2;117_11758_011.3;&outLink=Y#117_11758_011.3) ("스트레스 인지율" 선택)

두는 곳 — **로컬 Jupyter**: 이 노트북과 같은 폴더 / **Colab**: `/content/`에 업로드.  
컬럼 설명·함정 등 자세한 내용은 [`data/README.md`](data/README.md) 참고.

---

## 1. 그냥 읽어보기 — 어, 에러가 나네?

한국 공공데이터를 받아서 그냥 읽으면 흔히 이런 에러가 나요.


In [12]:
import pandas as pd

try:
    df = pd.read_csv('data/경찰청_범죄 발생 지역별 통계.csv')
    print(df.head())
except UnicodeDecodeError as e:
    print(f"{type(e).__name__}: {e}")


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xb9 in position 0: invalid start byte


`UnicodeDecodeError`는 "이 파일을 UTF-8로 못 읽음"이라는 뜻이에요.  
한국 공공데이터는 보통 **cp949(EUC-KR)** 인코딩이라 UTF-8로 읽으면 깨져요.

> 💡 **두 가지 표준 인코딩**:
> - **cp949** (Windows 한국어): 한국 공공데이터 대부분 (교육부, 행안부 등)
> - **utf-8**: 국제 표준, KOSIS 등 일부 기관


## 2. 인코딩 지정해서 다시 읽기


In [11]:
# encoding='cp949' 추가
df = pd.read_csv('data/경찰청_범죄 발생 지역별 통계.csv', encoding='cp949')
df.head()

,범죄대분류,범죄중분류,서울 종로구,서울 중구,서울 용산구,서울 성동구,서울 광진구,서울 동대문구,서울 중랑구,서울 성북구,...,외국 러시아,외국 튀르키예,외국 중국,외국 일본,외국 필리핀,외국 베트남,외국 태국,외국 말레이시아,외국 인도네시아,외국 기타국가
0,강력범죄,살인기수,0,3,5,2,1,2,3,1,...,0,0,0,0,1,0,0,0,0,1
1,강력범죄,살인미수등,1,2,5,3,1,5,2,1,...,0,0,1,0,1,2,0,0,0,0
2,강력범죄,강도,5,5,4,2,6,4,2,1,...,0,0,0,0,2,0,1,0,0,3
3,강력범죄,강간,31,21,52,22,49,25,35,32,...,0,0,1,7,2,1,3,0,1,2
4,강력범죄,유사강간,6,7,9,3,9,9,5,8,...,0,0,0,0,0,1,0,0,0,0


**해결!** 한글이 깨지지 않고 잘 들어왔어요.


## 3. 다른 데이터 — utf-8 사례

KOSIS 같은 기관은 utf-8을 쓰기도 해요. 같은 코드로 다른 데이터 읽어보면:


In [13]:
# 이 파일은 utf-8이에요 (KOSIS 데이터)
stress = pd.read_csv('data/스트레스_인지율.csv', encoding='utf-8')  # 또는 그냥 안 줘도 OK (utf-8이 기본)
print(f"shape: {stress.shape}")
stress.iloc[:3, :5]


shape: (24, 235)


,시점,전체,전체.1,전체.2,전체.3
0,시점,소계,소계,소계,소계
1,시점,전체,전체,전체,남학생
2,시점,분석대상자수 (명),인지율 (%),표준오차,분석대상자수 (명)


- `encoding`을 명시 안 하면 pandas는 utf-8을 가정해요.  
- 그래서 utf-8 파일은 옵션 없이 잘 읽히고, cp949 파일은 에러 나요.


## 4. 인코딩 자동 탐지 (보너스)

받은 파일이 어떤 인코딩인지 모를 때 — `chardet` 라이브러리로 확인 가능.


In [17]:
# Colab에는 chardet가 기본 설치되어 있어요
# 로컬 환경에서는 설치 필요: pip install chardet
import chardet

def detect_encoding(filepath, n_bytes=10000):
    with open(filepath, 'rb') as f:
        raw = f.read(n_bytes)
    result = chardet.detect(raw)
    return result

# 두 파일 비교
for filename in ['data/경찰청_범죄 발생 지역별 통계.csv', 'data/스트레스_인지율.csv']:
    result = detect_encoding(filename)
    print(f"{filename}")
    print(f"  → {result['encoding']} (확신도: {result['confidence']:.0%})")
    print()


data/경찰청_범죄 발생 지역별 통계.csv
  → CP949 (확신도: 13%)

data/스트레스_인지율.csv
  → utf-8 (확신도: 99%)



## 5. 본인 데이터에 적용해보기 ✏️

본인이 받은 CSV에 다음 순서로 시도해보세요:

1. 그냥 `pd.read_csv(파일)` 시도
2. UnicodeDecodeError 나면 → `encoding='cp949'` 추가
3. 그래도 안 되면 → `chardet`로 확인

```python
# 본인 파일 경로로 바꿔서 시도
my_file = '본인_파일명.csv'

try:
    df = pd.read_csv(my_file)
except UnicodeDecodeError:
    try:
        df = pd.read_csv(my_file, encoding='cp949')
    except UnicodeDecodeError:
        # chardet으로 확인
        import chardet
        with open(my_file, 'rb') as f:
            enc = chardet.detect(f.read(10000))['encoding']
        df = pd.read_csv(my_file, encoding=enc)
```


## 6. 주의사항

### 6.1 BOM 포함된 utf-8
- 일부 파일은 utf-8 앞에 BOM(Byte Order Mark)가 붙어 있음
- 증상: 첫 컬럼명이 `'\ufeff시도'`처럼 이상한 문자가 붙음
- **해결**: `encoding='utf-8-sig'` 사용

### 6.2 엑셀 파일 (.xlsx)
- `pd.read_csv`로 못 읽음 → `pd.read_excel` 사용
- 인코딩 신경 안 써도 됨 (엑셀 형식이 알아서 처리)

### 6.3 인코딩 잘못 잡아도 에러 안 나는 경우
- ASCII만 있는 파일은 어떤 인코딩으로 읽어도 에러 안 남
- 하지만 한글이 깨질 수 있음 → **head()로 한글 확인**


## 7. 📚 더 알아보기

- `encoding='euc-kr'` — cp949와 거의 같지만 약간 다름
- Tab 구분 파일: `pd.read_csv(..., sep='\t')` 또는 `pd.read_table`
